In [69]:
import os
import sys

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [70]:
from dotenv import load_dotenv

In [72]:
from pydantic import BaseModel, Field
from typing import List, Literal

#News content Schema
class StockNews(BaseModel):
    ticker: str = Field(description="Stock Ticker (e.g: APPL, TSLA)")
    content: str = Field(description="Stock news ")
    url: str = Field(description="stock news url")




In [73]:
from pydantic import BaseModel
class StockNewsItem(BaseModel):
    ticker: str
    company_name: str
    title: str
    content: str
    url: str
    published_date: str

In [ ]:
from tavily import TavilyClient

tavily_client = TavilyClient();

def search_news(ticker: str, company_name: str, max_results: int) -> list[StockNewsItem]:
    """return the most recent news limited by number"""
    
    query = f'{company_name} ({ticker}) latest financial news article'

    response = tavily_client.search(query=query, 
                                    topic="news", 
                                    max_results=max_results,
                                    search_depth="basic",
                                    include_domains=["bloomberg.com","reuters.com","wsj.com","ft.com","cnbc.com","finance.yahoo.com"])
    
    results = [StockNewsItem(
        ticker=ticker,
        company_name=company_name,
        title=r['title'],
        content=r['content'],
        url=r['url'],
        published_date=r['published_date']
    ) for r in response['results']]
   
    return results



In [75]:
ticker = "MU"
from app.utils.financial import get_company_name
company_name = get_company_name(ticker)
print(company_name)

results = search_news(ticker, company_name, 3)

Micron Technology, Inc.


In [76]:
type(results[0])

__main__.StockNewsItem

In [85]:
from IPython.display import display,Markdown


for r in results:
    md_content = f"""
### {r.company_name} ({r.ticker}) news 
                
*{r.title}*
<br><br>
**Content:** {r.content}
<br><br>
[{r.title}]({r.url})
<br><br>
"""
    display(Markdown(md_content))


### Micron Technology, Inc. (MU) news 

*Micron hits $1 trillion market cap for the first time as stock surges 18% - CNBC*
<br><br>
**Content:** # Micron hits $1 trillion market cap for the first time as stock surges 18%. * Micron topped a $1 trillion market value for the first time on Tuesday. * Shares popped after UBS nearly tripled its price target on the stock. * Micron is among a new group of chipmakers benefitting from the shift to agentic AI. Micron topped a $1 trillion market value for the first time on Tuesday as shares popped 18%, driven by insatiable artificial intelligence demand for its memory chips. The stock surge came as UBS nearly tripled its price target on the stock from $535 to $1,625 a share, citing long-term agreement opportunities with partially fixed pricing. "We believe the market will start to put a more 'normal' multiple on the stock and MU will continue to re-rate higher as more details emerge about the structural changes AI has driven to the entire memory complex," the firm wrote. Micron's stock has more than tripled year to date.
<br><br>
[Micron hits $1 trillion market cap for the first time as stock surges 18% - CNBC](https://www.cnbc.com/2026/05/26/micron-stock-trillion-market-cap.html)
<br><br>



### Micron Technology, Inc. (MU) news 

*Micron Gets Higher Target From DA Davidson - Yahoo Finance*
<br><br>
**Content:** - Best gifts for sister-in-law. - Best birthday gifts for her. - Best get well soon gifts. - Best gifts for mother-in-law. - Best anniversary gifts for wife. ### New on Yahoo. # Micron Gets Higher Target From DA Davidson. Micron Technology (MU, Financials) received another bullish call as DA Davidson raised its price target on the memory-chip maker to $1,500 from $1,000. The firm kept its Buy rating, pointing to a shift in how investors may need to think about Micron's business. That makes Micron's role more important and potentially less interchangeable than in past memory cycles. Micron has already rallied sharply this year and recently reached a $1 trillion market value for the first time. For investors, the question is whether AI demand can keep memory pricing strong long enough to support higher earnings and cash flow. The next test will be whether Micron continues to show that this cycle is structurally different from past memory booms.
<br><br>
[Micron Gets Higher Target From DA Davidson - Yahoo Finance](https://finance.yahoo.com/markets/stocks/articles/micron-gets-higher-target-da-221924075.html)
<br><br>



### Micron Technology, Inc. (MU) news 

*Micron is one of the most overbought stocks after this week’s rally to new highs - CNBC*
<br><br>
**Content:** # Micron is one of the most overbought stocks after this week’s rally to new highs. Micron Technology emerged as one of the most overbought stocks this week as the stock market rallied to new record highs. Stocks ended the week higher, boosted by gains in the technology sector and optimism over a ceasefire extension in the Middle East, and all three major market averages scored new intraday and closing records on Friday. Stocks with a 14-day RSI above 70 are considered overbought, meaning that a pullback could be on the horizon. The following table shows this week's most overbought names: Insatiable demand for Micron's memory chips in artificial intelligence applications drove its stock 29% higher this week. "We believe the market will start to put a more 'normal' multiple on the stock and MU will continue to re-rate higher as more details emerge about the structural changes AI has driven to the entire memory complex," UBS analyst Timothy Arcuri wrote in a note to clients.
<br><br>
[Micron is one of the most overbought stocks after this week’s rally to new highs - CNBC](https://www.cnbc.com/2026/05/30/micron-is-one-of-the-most-overbought-stocks-after-this-weeks-rally-to-new-highs.html)
<br><br>


In [ ]:
#define a graph
from typing import List, Optional
from pydantic import BaseModel, Field

class NewsItems(BaseModel):
    title: str
    content: str
    content_kr: str
    sumamry: str
    summary_kr: str
    published_date: str
    url: str
    sentiment_lable: str
    sentiment_score: float


class StockNewsState(BaseModel):
    ticker: str = Field(description="Ticker symbol. e.g. SNDK, APPL")
    company_name: str = Field(description="The company name from the ticker symbol. e.g. Apple, inc.")
    news_items: Optional[List[NewsItems]] = Field(default_factory=list)


In [ ]:
def load_news(state):
    """Load news from data base if news exist; fill the state"""
    return {}

def route_messages(state):
    """if there are news items in the state go to END; otherwise, go grab news!"""
    return 'search_news'

def search_news(state):
    """Search Top 3 news"""
    return {}

def summarize_news(state):
    """Summarize 3 news and add sentiment and embedding"""
    return {}

def save_to_db(state):
    """Save the data to supadb"""
    return {}

In [ ]:
from IPython.display import Image, display
from langgraph.graph import StateGraph, START, END
builder = StateGraph(StockNewsState)
builder.add_node("load_news", load_news)
builder.add_node("route_messages", route_messages)
builder.add_node("search_news", search_news)
builder.add_node("summarize_news", summarize_news)
builder.add_node("save_to_db", save_to_db)

builder.add_edge(START, "load_news")
builder.add_edge("load_news","route_messages")
builder.add_conditional_edges("route_messages", route_messages, ["search_news", END])
builder.add_edge("search_news","summarize_news")
builder.add_edge("summarize_news", "save_to_db")
builder.add_edge("save_to_db", END)

stock_news_graph = builder.compile()

display(Image(stock_news_graph.get_graph(xray=1).draw_mermaid_png()))


ValueError: Found edge starting at unknown node 'route_messages'